In [0]:
# Bronze ingestion: ALL leads Excel files from /Volumes/b2b_ecommerce/raw/leads -> bronze.leads streaming table.
# Auto Loader reads each .xlsx as raw bytes (binaryFile) so no file is skipped; openpyxl parses each
# workbook SHEET BY SHEET starting at each sheet's own header row (the first row with >= 3 filled
# cells), so workbooks that carry 'Marketing leads export' title rows above the header are handled
# like any other file, and data on any sheet is read, not just the first.
# The output is a proper table with the files' 22 columns, one per column, every value kept as raw
# text, plus tracking columns and _rescued_data: a JSON string holding any column a file carries
# beyond the 22 expected ones, so nothing is silently dropped. mapInPandas must declare its output
# schema (Spark API constraint); the list below is exactly the header every workbook carries.
# Failure behaviour: a file that cannot be opened or parsed is SKIPPED - logged at ERROR level with
# the file name and reason - and the update continues with the healthy files; one bad file never
# fails the whole run. Schema drift never fails either: extra columns land in _rescued_data,
# missing columns load as NULL, and a header mismatch is flagged by the warn-level expectation on
# lead_id (SELECT source_file FROM bronze.leads WHERE lead_id IS NULL finds the offending file).
# Note: each workbook is parsed fully in memory on one worker - fine for the current ~KB-sized
# files; watch file sizes if workbooks grow.
import io
import json
import logging
import os
from datetime import datetime, timezone
from typing import Iterator

import pandas as pd
from openpyxl import load_workbook

from pyspark import pipelines as dp
from pyspark.sql.types import (
    BinaryType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

logger = logging.getLogger("bronze.leads")

SOURCE_PATH = "/Volumes/b2b_ecommerce/raw/leads"

BINARY_FILE_SCHEMA = StructType(
    [
        StructField("path", StringType()),
        StructField("modificationTime", TimestampType()),
        StructField("length", LongType()),
        StructField("content", BinaryType()),
    ]
)

# The 22 source columns, exactly as they appear in the Excel header row.
LEADS_COLUMNS = [
    "lead_id",
    "created_date",
    "source",
    "campaign_name",
    "utm_source",
    "utm_medium",
    "utm_campaign",
    "country",
    "city",
    "company_name",
    "company_size",
    "industry",
    "lead_score",
    "estimated_order_volume",
    "status",
    "utm_content",
    "contact_name",
    "contact_email",
    "phone",
    "job_title",
    "sales_rep",
    "last_contacted_date",
]

OUTPUT_SCHEMA = (
    ", ".join(f"{c} STRING" for c in LEADS_COLUMNS)
    + ", source_file STRING, ingested_at TIMESTAMP, updated_at TIMESTAMP, _rescued_data STRING"
)


def _parse_workbook(content: bytes, path: str) -> list:
    """Return one dict per data row across ALL sheets, starting at each sheet's header row.

    Every value is kept as raw text. Raises ValueError naming the file and the problem;
    the caller logs it and skips the file.
    """
    name = os.path.basename(path)
    try:
        wb = load_workbook(io.BytesIO(content), read_only=True, data_only=True)
    except Exception as e:
        raise ValueError(f"{name}: cannot open workbook - {type(e).__name__}: {e}") from None
    records = []
    try:
        for sheet in wb.worksheets:
            rows = list(sheet.iter_rows(values_only=True))
            try:
                hidx = next(
                    i for i, r in enumerate(rows[:5]) if sum(v is not None for v in r) >= 3
                )
            except StopIteration:
                if any(any(v is not None for v in r) for r in rows):
                    logger.warning(
                        "%s: sheet '%s' skipped - no header row (>= 3 filled cells) in its first 5 rows",
                        name,
                        sheet.title,
                    )
                continue
            header = [str(v).strip() for v in rows[hidx]]
            sheet_records = [
                {h: None if v is None else str(v) for h, v in zip(header, r)}
                for r in rows[hidx + 1 :]
                if not all(v is None for v in r)
            ]
            logger.debug(
                "%s: sheet '%s': header at row %d, %d columns, %d data rows",
                name,
                sheet.title,
                hidx + 1,
                len(header),
                len(sheet_records),
            )
            records.extend(sheet_records)
    finally:
        wb.close()
    return records


def _parse_leads_files(batches: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    """mapInPandas batch function: parse every workbook, skip failures, log a batch summary."""
    expected = set(LEADS_COLUMNS)
    for batch in batches:
        records = []
        files_parsed = 0
        files_skipped = []
        for path, content in zip(batch["path"], batch["content"]):
            try:
                data = _parse_workbook(content, path)
            except ValueError as e:
                logger.error("skipping file - %s", e)
                files_skipped.append(str(e))
                continue
            files_parsed += 1
            now = datetime.now(timezone.utc)
            for record in data:
                rescued = {k: v for k, v in record.items() if k not in expected}
                records.append(
                    {
                        **{c: record.get(c) for c in LEADS_COLUMNS},
                        "source_file": path,
                        "ingested_at": now,
                        "updated_at": now,
                        "_rescued_data": json.dumps(rescued) if rescued else None,
                    }
                )
        out = pd.DataFrame(
            records,
            columns=LEADS_COLUMNS
            + ["source_file", "ingested_at", "updated_at", "_rescued_data"],
        )
        logger.info(
            "batch: %d file(s) parsed -> %d row(s); %d file(s) skipped",
            files_parsed,
            len(records),
            len(files_skipped),
        )
        yield out


@dp.table(
    name="bronze.leads",
    comment="Raw leads from Excel files in /Volumes/b2b_ecommerce/raw/leads; every file and sheet handled from its header row, values kept as raw text, unknown columns preserved in _rescued_data.",
)
# Warn-level check: a workbook whose header row does not match the expected columns produces rows
# with NULL in every source column, so a NULL lead_id flags a header mismatch. Rows are kept -
# bronze stays raw - but the violation is counted in the pipeline's data quality metrics and can be
# traced to its file via: SELECT source_file FROM bronze.leads WHERE lead_id IS NULL.
@dp.expect("header_matches_leads_schema", "lead_id IS NOT NULL")
def leads():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .schema(BINARY_FILE_SCHEMA)
        .load(SOURCE_PATH)
        .mapInPandas(_parse_leads_files, OUTPUT_SCHEMA)
    )